# FARGO Operator Checkpoint Reproduction Demo

This notebook demonstrates how to inspect the saved checkpoints and rerun rollout evaluation when the full FARGO dataset is available. It is intentionally lightweight: checkpoint/metric inspection works without the large dataset; rollout cells require the memmap dataset.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.fargo_operator.checkpoints import checkpoint_summary

BASELINE_ROOT = ROOT / 'artifacts/baselines'
MAIN_LOCAL_CKPT = ROOT / 'artifacts/main_rollout/plain_fno_checkpoint.pkl'
DATASET = ROOT / 'data/fargo_transient_10orbits_128f'
BASELINE_ROOT, DATASET


## 1. Inspect Available Baseline Checkpoints

The baseline checkpoints should live under `artifacts/baselines/`. The loader handles NumPy 2.x checkpoints in NumPy 1.x environments.


In [ ]:
def read_metrics(ckpt_path):
    metrics_path = ckpt_path.with_name('metrics.json')
    if not metrics_path.exists():
        return {}
    data = json.loads(metrics_path.read_text())
    model = ckpt_path.parent.name
    result = data.get('results', {}).get(model, {})
    return {
        'val_rmse': result.get('validation', {}).get('rmse'),
        'hpt_rmse': result.get('heldout_parameter_time', {}).get('rmse'),
        'rollout_rmse': result.get('rollout', {}).get('rmse'),
        'speed_ms_per_batch': result.get('speed_ms_per_batch'),
    }

rows = []
for ckpt in sorted(BASELINE_ROOT.glob('*/*_checkpoint.pkl')):
    s = checkpoint_summary(ckpt)
    rows.append({
        'model': s['model'],
        'checkpoint': str(ckpt.relative_to(ROOT)),
        'dataset': s['dataset'],
        'width': s['model_config'].get('width'),
        'depth': s['model_config'].get('depth'),
        'modes_r': s['model_config'].get('modes_r'),
        'modes_theta': s['model_config'].get('modes_theta'),
        **read_metrics(ckpt),
    })
rows


In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(rows).sort_values('hpt_rmse')
    display(df)
except Exception:
    for row in sorted(rows, key=lambda r: r.get('hpt_rmse') or 999):
        print(row)


## 2. Main Experiment Checkpoint Location

The main long-rollout Plain 2D FNO checkpoint should live at `artifacts/main_rollout/plain_fno_checkpoint.pkl`.


In [ ]:
print('Recommended local checkpoint:', MAIN_LOCAL_CKPT.relative_to(ROOT))
print('Local main checkpoint exists:', MAIN_LOCAL_CKPT.exists())

if MAIN_LOCAL_CKPT.exists():
    checkpoint_summary(MAIN_LOCAL_CKPT)


## 3. Optional Rollout Evaluation

This cell runs a small rollout evaluation only when `data/fargo_transient_10orbits_128f` exists. On the server, switch `--jax-platform cpu` to `--jax-platform cuda` and increase `--eval-batches`.


In [ ]:
import subprocess

ckpt = MAIN_LOCAL_CKPT if MAIN_LOCAL_CKPT.exists() else BASELINE_ROOT / 'plain_fno/plain_fno_checkpoint.pkl'
cmd = [
    sys.executable, str(ROOT / 'scripts/evaluate_fargo_checkpoint_rollouts.py'),
    '--checkpoint', str(ckpt),
    '--dataset', str(DATASET),
    '--output-dir', str(ROOT / 'results/notebook_demo_rollouts'),
    '--horizons', '8', '16',
    '--batch-size', '1',
    '--eval-batches', '2',
    '--no-physics-metrics',
    '--jax-platform', 'cpu',
]
print(' '.join(cmd))
if DATASET.exists():
    subprocess.run(cmd, check=True, cwd=ROOT)
else:
    print('Skipping rollout evaluation because dataset is missing:', DATASET)


## 4. High-Resolution External Validation Command

For the high-resolution six-case validation used in the response/report, the server command was:

```bash
JAX_PLATFORMS=cuda python scripts/evaluate_fargo_checkpoint_rollouts.py \
  --checkpoint artifacts/main_rollout/plain_fno_checkpoint.pkl \
  --dataset data/hires6_eval_view \
  --output-dir results/hires6_eval_plain_fno_rollouts \
  --horizons 8 16 32 64 \
  --batch-size 1 \
  --eval-batches 12 \
  --no-physics-metrics \
  --jax-platform cuda
```

Reported relL2 results: rollout@8 = 4.0845%, rollout@16 = 5.3909%, rollout@32 = 8.0821%, rollout@64 = 11.8092%.
